# Experiment 4.2 — Continuous recurrent controls

Question: is the Exp4.0 gap caused by shared recurrent sequence modeling itself, or specifically by spiking dynamics / WholeCount-style readout?

Controls: bias-free tanh FF-ANN and vanilla RNN, widths 64/128, trained with endpoint CE or valid summed-logit CE on the exact same scaled Fixed250 input.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'snn').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate writingRing repository root')

repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from scripts import experiment_4_2_continuous_recurrent_controls as exp42

root = exp42.results_dir(repo_root)
runs_path = root / 'runs.csv'
summary_path = root / 'summary.csv'
references_path = root / 'references.csv'
eval_dir = root / 'evaluations'
eval_count = len(list(eval_dir.glob('*.json'))) if eval_dir.exists() else 0
ready = runs_path.exists() and summary_path.exists() and references_path.exists()
print('Artifact root:', root)
print(f'Continuous-control evaluations: {eval_count}/{exp42.EXPECTED_RUNS}')
print('Finalized:', ready)
if ready:
    runs = pd.read_csv(runs_path)
    summary = pd.read_csv(summary_path)
    references = pd.read_csv(references_path)
    display(summary)
    display(references)
else:
    runs = summary = references = None
    print('Run: bash scripts/bash_script/SNN_Bash/submit_exp_4_2_cpu.bash')

In [ ]:
if not ready:
    print('Skipped: continuous-control plot requires finalized artifacts.')
else:
    test = runs[runs['split'] == 'test'].copy()
    view = (test.groupby(['model_type', 'hidden_width', 'objective'], as_index=False)
                ['primary_balanced_accuracy'].mean())
    labels = [f"{r.model_type} H={int(r.hidden_width)} {r.objective}" for r in view.itertuples()]
    fig, ax = plt.subplots(figsize=(11, 5.5))
    ax.bar(labels, view['primary_balanced_accuracy'])
    linear_ba = references.loc[references['method'] == 'linear', 'test_balanced_accuracy'].iloc[0]
    ax.axhline(linear_ba, linestyle='--', label='Fixed250 + Linear')
    ax.set_ylabel('Mean test balanced accuracy')
    ax.set_title('Continuous temporal decoders vs Linear reference')
    ax.tick_params(axis='x', rotation=45)
    ax.legend()
    ax.grid(True, axis='y', alpha=0.25)
    plt.tight_layout()
    plt.show()

In [ ]:
if not ready:
    print('Skipped: objective/readout comparison requires finalized artifacts.')
else:
    test = runs[runs['split'] == 'test'].copy()
    cols = ['model_type', 'hidden_width', 'objective', 'primary_balanced_accuracy',
            'endpoint_logits_balanced_accuracy', 'valid_sum_logits_balanced_accuracy',
            'full_sum_logits_balanced_accuracy', 'tail_logit_l1_fraction']
    display(test.groupby(['model_type', 'hidden_width', 'objective'])[cols[3:]].agg(['mean', 'std']))
    display(references.groupby(['method', 'hidden_width', 'tau_mem_ms'], dropna=False)
            [['test_balanced_accuracy', 'test_macro_f1']].agg(['mean', 'std']))